# PyPhi

> Mayner WGP, Marshall W, Albantakis L, Findlay G, Marchman R, Tononi G. [PyPhi: A toolbox for integrated information theory](https://doi.org/10.1371/journal.pcbi.1006343). PLOS Computational Biology 14(7): e1006343. 2018.

## Prepare environment

[Latest](https://pypi.org/p/pyphi) published version requires **Python 3.7**.

### Install on [Conda](https://docs.anaconda.com/miniconda/miniconda-other-installer-links/)

In [ ]:
# conda create -n pyphi python=3.7
# conda activate pyphi
# conda install graphviz==0.13 pygraphviz==1.3
# !pip install graphviz==0.13 matplotlib==3.1.1 networkx==2.3 numpy==1.16.2 plotly==3.10.0 pydot==1.4.1 pyphi==1.2.0

### Install on [Colab](https://colab.research.google.com/)

In [ ]:
# from IPython.display import clear_output
# !sudo apt-get update -y
# !sudo apt-get install python3.7
# !sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.7 1
# !sudo update-alternatives --config python3
# !sudo apt install python3-pip python3.7-distutils
# !pip install graphviz==0.13 matplotlib==3.1.1 networkx==2.3 numpy==1.16.2 plotly==3.10.0 pydot==1.4.1 pyphi==1.2.0 pygraphviz==1.3
# clear_output()

## Load prerequesites

### Import libraries

In [ ]:
import json
import logging as log
import os
from shutil import move

# os.environ["PYPHI_WELCOME_OFF"] = "yes"

import graphviz as gv
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pygraphviz as pgv
import pyphi

RENDERERS = ['graphviz', 'pygraphviz', 'pydot', 'none']

### Wrapper class

In [ ]:
class PyPhi():

    def __init__(self):
        """ Initializes class. """

    def __call__(
        self,
        matrix,
        A=[],
        states=[],
        current_state=[],
        num_states=None,
        output_folder="artifacts",
    ) -> list:
        """
        Calculates phi and builds PyPhi network.

        Input parameters:
            * matrix: stochastic matrix containing transition values
                as a dictionary of keys or an array of arrays
            * A: adjacency matrix for connecting states
            * states: list of states (strings) to use as labels
            * current_state: observed states in the system
            * num_states: maximum number of observed states
        """
        if isinstance(matrix, dict):
            if not states:
                states = list(matrix.keys())[:num_states]
            matrix = list(matrix.values())[:num_states]

        if num_states:
            matrix = matrix[:num_states]
            A = [list(x[:num_states]) for x in A[:num_states]]
            states = states[:num_states]

        clusters = len(matrix).__str__()
        timeslices = len(matrix[0]).__str__()

        # State labels
        if not states:
            states = [str(i) for i in range(len(Q))]

        # Most recent state
        if not current_state:
            for x in matrix:
                current_state.append(x[-1])

        # Connectivity matrix
        if A == []:
            for i in range(len(matrix)):
                A.append([])
                for j in range(len(matrix)):
                    A[i].append(1)

        # Connected subgraphs
        sg = self.get_connected_states(A)

        # Influence matrix based on state transitions
        im = self.generate_influence_matrix(matrix, A)

        # Transition probability matrix
        tpm = self.generate_tpm(matrix, im)

        # Generate network with PyPhi
        network = pyphi.network.Network(tpm, cm=A)

        # Generate ALL COMPLEXES structures of the network
        # May take a lot of time depending on the number of clusters
        ac = pyphi.compute.network.all_complexes(network, current_state)

        # Sort complexes by phi value and select
        # The complex with the greatest value
        ac.sort(key=self.get_phi, reverse=True)
        bc = ac[0] # <-- best complex

        # Best complex mechanisms and phi values
        ces = bc.ces

        # Store PyPhi results as output string
        output = str("Clusters: "+str(clusters)+
                    "\nTimeslices: "+str(timeslices)+
                    "\n\nStates:\n"+str(states)+
                    "\n\nPresence Matrix:\n"+str(np.array(matrix))+
                    "\n\nCurrent state:\n"+str(current_state)+
                    "\n\nAdjacency Matrix:\n"+str(np.array(A))+
                    "\n\nConnected states:\n"+str(sg)+
                    "\n\nInfluence Matrix:\n"+str(np.array(im)).replace("None","-")+
                    "\n\nTransition Probability Matrix:\n"+str(np.array(tpm))+
                    "\n\nBest complex mechanisms and phi values:")

        for i in range(len(ces)):
            output += str("\n( %s ) Mechanism = %s φ = %s" % (i, list(ces[i].mechanism), ces[i].phi))

        # Write to output file
        with open(f"{output_folder}/pyphi.txt", "w") as f:
            f.write(output)

        # Print to standard output
        print(output)

        # Write base view JSON file for PyPhi
        # Self.create_view_base(bc, states, current_state, A)

        return bc, ac

    def complex_mechanism(self, c, m=None, output_folder="artifacts"):
        """
        Displays complex (c) cause-effect structure from
        mechanism (m) in a slightly optimized plotly view.
        """
        c = c.ces # Cause effect-structure

        if m is None:
            c = sorted(c, key=lambda x: x.phi, reverse=True)
            m = 0

        c = c[m]  # Complex mechanism
        v = ["cause","effect"]

        for i in v:
            # Cause and effect
            print("\n---\n"+i.capitalize()+":\n---\n")

            value = getattr(c,i)
            matrix = len(list(value.purview))
            x = []
            y = []

            #print(value)
            print("\nMaximally-irreducible", i, "φ =", value.phi)
            print("\nMechanism:", value.mechanism)
            print("Purview:", value.purview)
            print("\nMIP:\n"+ str(value.mip))
            print("\nRepertoire:")

            purviewSize = len(list(value.purview))
            array = list(value.repertoire.reshape(1,pow(2,purviewSize))[0])

            for j in range(len(array)):
                binArray = self.create_bin_array(j,purviewSize)
                binArrayString = "".join(map(str, binArray))
                x.append("state-" + binArrayString)
                y.append(array[j])

            data = [go.Bar(
                x=x,
                y=y
                )]

            layout = go.Layout(
                autosize=False,
                width=800,
                height=300,
                title=f"Repertoire: Maximally-irreducible {i} φ = {value.phi} (Mechanism: {value.mechanism})",
                #yaxis=dict(8
                #range=[0,1])
                )

            fig = go.Figure(data=data, layout=layout)
            fig.write_html(f"{output_folder}/pyphi-{i}.html")
            fig.write_image(f"{output_folder}/pyphi-{i}.png")
            fig.show(renderer="colab")
            # Py.iplot(fig)

    def create_view_base(self, complex, states, current_state, A):
        '''
        Writes output JSON file for PyPhi view.

        Input parameters:
            * complex: to get subsystems
            * states: as a list of strings
            * current_state: in current tX
            * A: adjacency matrix
        '''
        subsystem = complex.subsystem.__str__()[10:-1].replace(" ","").split(',')
        nodes = self.get_graph_nodes(states, subsystem, current_state)
        edges = self.get_graph_edges(states, A)

        complexJSON = {

            "states": states,

            "graph": {"nodes": nodes,
                      "edges": edges},

            'sia': {'bigPhi': complex.phi,
                    'subsystem': complex.subsystem.__str__(),
                    'ces': []}}

        ces = complex.ces

        for i in ces:
            obj = self.get_CES_JSON_object(i)
            complexJSON["sia"]["ces"].append(obj)

        with open('view.json', 'w') as j:
            json.dump(complexJSON, j)

    def distance(self, o1, o2):
        """
        Returns centroid distance.
        """
        return 1 - self.semantic_similarity_number(o1, o2)

    def filter_connected_states(self, P, A=[], states_to_remove=[]):
        """
        Returns a subset of connected states in matrices.
        """
        if not states_to_remove:
            return (P, A)

        # Get list of states to remove
        if isinstance(states_to_remove, str):
            states_to_remove = states_to_remove.split(",")

            try:
                states_to_remove = [int(x) for x in states_to_remove]
            except ValueError:
                if isinstance(P, dict):
                    states_to_remove = [list(P.keys()).index(x) for x in states_to_remove]
                else:
                    raise TypeError("expected a dictionary to match list of states as keys")

        # Check list of states to remove
        if not isinstance(states_to_remove, list):
            raise TypeError("expected a list of states (indices or dictionary keys) as input")

        keys = list(P.keys())
        P = {keys[i]:value for i, value in enumerate(P.values()) if i not in states_to_remove}
        return (P, self.filter_matrix(A, states_to_remove) if A else [])

    def filter_matrix(self, matrix, states_to_remove=[]):
        """
        Returns matrix without specified items to remove.
        """
        return [[v for k, v in enumerate(matrix[i]) if k not in states_to_remove] for i in range(len(matrix)) if i not in states_to_remove]

    def generate_distances_matrix(self, centroids, threshold="median"):
        """
        Returns adjacency matrix based on centroids distance (median/mean):
        * if higher than threshold, nodes will not be connected (0);
        * if lower or equal, nodes will be connected (1).
        Note that a centroid will always be nearest its own cluster.
        """
        n = len(centroids)
        matrix = np.zeros(shape=(n,n))
        values = []

        for i,_ in enumerate(centroids):
            for j in range(int(i), len(centroids)):
                d = self.distance(centroids[i], centroids[j])
                values.append(d)
                matrix[i][j] = d
                matrix[j][i] = d

        if threshold:
            adjacency = np.zeros(shape=(n,n), dtype=np.int8)
            values = sorted(values)

            if threshold == "median":
                threshold = np.median(values)
            elif threshold == "mean":
                threshold = np.mean(values)
            else: # Error
                log.warning("THRESHOLD should be either 'mean' or 'median'.")
                return matrix

            for i,_ in enumerate(centroids):
                for j,_ in enumerate(centroids):
                    if matrix[i][j] <= threshold:
                        adjacency[i][j] = 1
                    else: adjacency[i][j] = 0

            return adjacency.tolist()

        return matrix

    def generate_influence_matrix(self, M, cm=[], normalized=True):
        """
        Returns matrix based on state transitions from each node to another:
        * influence equals the amount of hits a state has in t+1;
        * optionally considers only linked states in connectivity matrix (cm);
        * if normalized and NxN shaped, the returned matrix will be stochastic.
        """
        matrix = []

        for group in range(len(M)):
            matrix.append([])
            for target in range(len(M)):
                hits = 0
                if cm == [] or cm[group][target] == 1:
                    for i in range(1, len(M[group])):
                        if M[group][i] == M[target][i-1]:
                            hits += 1
                matrix[group].append(hits)

        if normalized:
            for i, array in enumerate(matrix):
                s = sum([(0 if x==None else x) for x in array])
                a = [round((0 if x in (0,None) else x/s), 2) for x in array]
                # Sum of each row must amount 1.0 (unity)
                a[a.index(max(a))] = round(max(a) + (1-sum(a)), 100)
                matrix[i] = a

        matrix = np.array(matrix)
        matrix = np.around(matrix, 2)

        return matrix.tolist()

    def generate_presence_matrix(self,  themes, by="median", output_folder="artifacts"):
        """ Convert an artifact to a presence matrix composed of binary data. """
        os.makedirs(output_folder) if not os.path.isdir(output_folder) else None

        matrix = {}
        threshold = None

        dates = sorted(set(x for x in themes.keys() for x in themes[x]))

        for label, dct in themes.items():
            matrix[label] = [len(dct[date]) if dct.get(date) else 0 for date in dates]

            if by == "median":
                threshold = np.median(matrix[label])
            elif by == "mean":
                threshold = np.mean(matrix[label])
            else:
                log.warning(f"Unrecognized threshold method (by='{by}').")

            if threshold:
                matrix[label] = [1 if v >= threshold else 0 for v in matrix[label]]

        with open(f"{output_folder}/matrix.json", "w") as j:
            json.dump(matrix, j)

        return matrix

    def generate_tpm(self, matrix, influence_matrix):
        """
        Builds Transition Probability Matrix for PyPhi.

        Input parameters:
            * matrix: input matching influence_matrix
            * influence_matrix: must be stochastic
        """
        tpm = []
        times = len(matrix[0])
        number_of_groups = len(matrix)
        #divisor = number_of_groups*(times-1)

        for i in range(pow(2, number_of_groups)):
            tpm.append([])
            bin_array = self.create_bin_array(i, number_of_groups)

            for group in range(len(matrix)):
                value = self.get_probability(group, bin_array, influence_matrix, number_of_groups, times-1)
                tpm[i].append(value)

        return tpm

    def generate_users_matrix(self, clusters, obj_array, threshold="median"):
        """
        Returns adjacency matrix based on users in clusters (median/mean):
        * if higher than or equal to threshold, nodes will be connected (1);
        * if lower, nodes will not be connected (0).
        """
        n = len(clusters)
        matrix = np.zeros(shape=(n,n))
        users = {}

        for c in clusters.keys():
            for obj in clusters[c]:
                screen_name = obj["screen_name"]
                try: users[screen_name].append(c)
                except: users[screen_name] = [c]

        users = [set(u) for u in users.values()]

        for u in users:
            for c in u:
                matrix[c][c] += 1
            for i,j in combinations(u, 2):
                matrix[i][j] += 1
                matrix[j][i] += 1

        if threshold:
            adjacency = np.zeros(shape=(n,n), dtype=np.int8)
            values = []

            for i,x in enumerate(matrix):
                adjacency[i][i] = matrix[i][i]
                for j,v in enumerate(x):
                    values.append(v)

            values = sorted(values)

            if threshold == "median":
                threshold = np.median(values)
            elif threshold == "mean":
                threshold = np.mean(values)
            else: # Error
                log.warning("THRESHOLD should be either 'mean' or 'median'.")
                return matrix

            for i,_ in enumerate(clusters):
                for j,_ in enumerate(clusters):
                    if i == j or matrix[i][j] >= threshold:
                        adjacency[i][j] = 1
                    else: adjacency[i][j] = 0

            return adjacency.tolist()

        return matrix

    def get_CES_JSON_object(self, ces):
        '''
        Returns cause-effect structure as a JSON object.
        '''
        cesJsonObject = {
            "smallphi": ces.phi,
            "mechanism": list(ces.mechanism),
            "cause":{
                "mip":ces.cause.mip.__str__(),
                "smallphi":ces.cause.phi,
                "purview":list(ces.cause.purview)},
            "effect":{
                "mip":ces.effect.mip.__str__(),
                "smallphi":ces.effect.phi,
                "purview":list(ces.cause.purview)}}
        # Cause repertoire array
        n = len(list(ces.cause.purview))
        cesJsonObject['cause']['repertoire'] = self.get_repertoire_array(ces.cause,n)
        # Effect repertoire array
        n = len(list(ces.effect.purview))
        cesJsonObject['effect']['repertoire'] = self.get_repertoire_array(ces.effect,n)
        # Returns cause effect-structure
        return cesJsonObject

    def get_connected_states(self, A, states=[]):
        """
        Returns connected states or subgraphs
        based on the input adjacency matrix *A*.
        """
        dim = len(A)
        subgraphs = []
        free_nodes = []

        for i in range(dim):
            free_nodes.append(i)

        while len(free_nodes) > 0:
            i = 0
            subgraph = []
            pivot = free_nodes.pop(0)
            subgraph.append(pivot)

            while i < len(subgraph):
                j = 0
                source = subgraph[i]

                while j < len(free_nodes):
                    target = free_nodes[j]
                    hit = A[source][target]

                    if hit == 1:
                        subgraph.append(free_nodes.pop(j))
                    else: j+=1

                i+=1

            subgraphs.append(subgraph)

        if states:
            return states if not subgraphs else [[states[x] for x in s] for s in subgraphs]

        return subgraphs

    def get_repertoire_array(self, ces, n):
        '''
        Return repertoire list.

        Input parameters:
            * ces: cause-effect structure
            * n: number of digits
        '''
        repertoire = []
        array = list(ces.repertoire.reshape(1,pow(2,n))[0])
        # Append key and value to list
        for i in range(len(array)):
            repertoire.append({
                "key": self.create_bin_array(i,n).__str__().replace("[","").replace(",","").replace("]","").replace(" ",""),
                "value": array[i]})
        # Return repertoire list
        return repertoire

    def get_phi(self, obj):
        """
        Sort all complexes in *obj* by their phi values.
        """
        return obj.phi

    def get_probability(self, group, bin_array, influence_matrix, number_of_groups, max_influence):
        """
        Returns probability of transition for
        each group based on influence matrix.

        Input parameters:
            * group: in influence matrix
            * bin_array: binary array
            * influence_matrix: must be stochastic
            * number_of_groups: to consider for P
            * max_influence: to consider for P
        """
        total = 0
        none_count = 0

        influence_list = influence_matrix[group]

        for i in range(len(influence_list)):
            if influence_list[i] == None:
                none_count += 1
            else:
                if bin_array[i] == 1:
                    total += influence_list[i]
                else:
                    total += (max_influence - influence_list[i])

        try: return total/((number_of_groups-none_count)*(max_influence if max_influence>0 else 1))
        except: return 0

    def load_json_matrix(self, input_json):
        """
        Loads *input_json* file containing data for PyPhi.

        Accepted formats from this file are listed below:

        * Matrix and connectivity matrix from dictionary:
            {"matrix": <dictionary of keys or array of arrays>,
            "A": <array of arrays>}

        * Matrix as a dictionary of keys:
            {"state1": [0,1,0,0,1,0],
            "state2": [1,0,1,1,0,0],
            "state3": [0,1,0,1,0,1]}

        * Matrix as an array of arrays:
            [[0,1,0,0,1,0],
            [1,0,1,1,0,0],
            [0,1,0,1,0,1]]

        * Connectivity matrix as an array of arrays:
            [[1,1,1],
            [1,1,1],
            [1,1,1]]
        """
        with open(input_json, "r") as j:
            data = json.loads(j.read())

        if "matrix" in data:
            matrix = data["matrix"]
            A = data["A"] if "A" in data else (data["cm"] if "cm" in data else [])
        else:
            matrix = self.generate_presence_matrix(data)
            A = []

        return matrix, A

    def mc_render(self, matrix, states=[], states_to_remove=[], renderer="graphviz", ext="png", prog="dot", output_folder="."):
        """
        Builds a state-transition diagram or markov chain.
        Output files written: "markov.dot" and "markov.png".

        Input parameters:
            * matrix: stochastic matrix containing transition values
                as a dictionary of keys or an array of arrays
            * states: list of states (strings) to use as labels
            * renderer: choose "graphviz"; "pygraphviz"; or "pydot"
            * ext: image file type format to write as (default: PNG)
            * prog: others may be available by graphviz e.g. "neato"
            * output_folder: output folder name to write files to
        """
        edge_labels = {}

        output_dot = f"{output_folder}/markov.dot"
        output_ext = f"{output_folder}/markov." + ext

        # Filter matrix states by user list
        if states_to_remove:
            matrix = self.mc_filter(matrix, states_to_remove, True)

        # Matrix as a dictionary of keys
        if isinstance(matrix, dict):
            if not states:
                states = list(matrix.keys())
            matrix = list(matrix.values())

        # Prepare graph object
        if renderer == "graphviz":
            G = gv.Digraph(format=ext)
            G.attr("node", shape="circle")
            G.attr(rankdir="LR", size="8")
            G.add_node = G.node
            G.add_edge = G.edge

        elif renderer == "pygraphviz":
            G = pgv.AGraph(format=ext, strict=False, directed=True)

        elif renderer == "pydot":
            G = nx.MultiDiGraph()

        elif renderer == "none":
            pass

        else: # Exit
            raise ValueError(f"Unrecognized renderer: '{renderer}'.\n"+
                             f"Available choices: 'graphviz', 'pygraphviz', 'pydot', 'none'.")

        # State labels
        if not states:
            states = [str(i) for i in range(len(matrix))]

        # Add state nodes
        for s in states:
            G.add_node(s)

        # Build graph network
        for i, origin_state in enumerate(states):
        # For i in range(len(matrix)):
            # Origin_state = states[i]
            for j, destination_state in enumerate(states):
            # For j in range(i+1, len(matrix)):
                # Destination_state = states[j]
                rate = matrix[i][j] # Im[i][j]
                if (rate != None and rate > 0):
                    str_rate = str("{:.02f}".format(rate))
                    G.add_edge(origin_state, # States[i],
                            destination_state, # States[j],
                            weight=rate if renderer != "graphviz" else None,
                            label=str_rate)
                    edge_labels[(origin_state, destination_state)] = label=str_rate

        # Create output path directory
        if not os.path.exists(output_folder):
            os.mkdir(output_folder)

        if renderer == "graphviz":
            G.render(f"{output_folder}/markov")
            move(f"{output_folder}/markov", output_dot)
            return gv.Source.from_file(output_dot)

        elif renderer == "pygraphviz":
            G.layout(prog=prog)
            G.draw(output_ext, format=ext, prog=prog)
            return G.draw(format=ext, prog=prog)

        elif renderer == "pydot":
            plt.axis("off")
            plt.tight_layout()
            pos = nx.nx_pydot.pydot_layout(G, prog=prog)
            nx.drawing.nx_pydot.write_dot(G, output_dot)
            nx.draw_networkx_nodes(G, pos, node_shape="s", node_color="w", node_size=1000)
            nx.draw_networkx_edges(G, pos, width=1.0, alpha=0.5)
            nx.draw_networkx_labels(G, pos, font_weight=2)
            nx.draw_networkx_edge_labels(G, pos, edge_labels)
            plt.savefig(output_ext, format=ext)
            return plt.plot()

        return G

    def mc_filter(self, matrix, states_to_remove=[], normalized=False):
        """
        Returns an optionally normalized matrix with selected rows.
        """
        if not states_to_remove:
            return matrix

        # Get list of states to remove
        if isinstance(states_to_remove, str):
            states_to_remove = states_to_remove.split(",")

            try:
                states_to_remove = [int(x) for x in states_to_remove]
            except ValueError:
                if isinstance(matrix, dict):
                    states_to_remove = [list(matrix.keys()).index(x) for x in states_to_remove]
                else:
                    raise TypeError("expected a dictionary to match list of states as keys")

        # Check list of states to remove
        if not isinstance(states_to_remove, list):
            raise TypeError("expected a list of states (indices or dictionary keys) as input")

        # Get matrix values
        array = matrix
        if isinstance(matrix, dict):
            array = list(matrix.values())

        # Filter matrix values
        output = self.normalize_matrix(self.filter_matrix(array, states_to_remove))\
                 if normalized else self.filter_matrix(array, states_to_remove)

        if isinstance(matrix, dict):
            keys = [list(matrix.keys())[i] for i in range(len(matrix.keys())) if i not in states_to_remove]
            return {keys[i]: output[i] for i in range(len(output))}

        return output

    @staticmethod
    def create_bin_array(number, digits):
        """
        Converts a number in a binary array with n digits, as in:
            create_bin_array(10,5) -> 01010
        """
        array = []
        binString = format(number,"b")

        for i in range(digits-len(binString)):
            array.append(0)

        for i in range(len(binString)):
            array.append(int(binString[i]))

        return array

    @staticmethod
    def get_graph_nodes(states, subsystem, state):
        '''
        Returns nodes in graph.

        Input parameters:
            * states: as a list of strings
            * subsystem: from complex
            * state: from subsystem
        '''
        nodes = []

        for i in range(len(states)):
            status = ""

            if states[i] in subsystem:
                status = "on" if (state[i] == 1) else "off"
            else:
                status = "undefined"

            nodes.append({
                "id":states[i],
                "status": status})

        return nodes

    @staticmethod
    def get_graph_edges(states, A):
        '''
        Returns edges from connected nodes.

        Input parameters:
            * states: as a list of strings
            * A: adjacency matrix
        '''
        edges = []
        for i in range(len(A)):
            for j in range(i+1,len(A)):
                if A[i][j] == 1:
                    edges.append({
                        "source":states[i],
                        "target":states[j]})

        return edges

    @staticmethod
    def normalize_matrix(matrix):
        """
        Returns normalized matrix in which the
        sum of each row amounts 1.0 (unity).
        """
        for i, array in enumerate(matrix):
            s = sum([(0 if x==None else x) for x in array])
            a = [round((0 if x in (0,None) else x/s), 2) for x in array]
            a[a.index(max(a))] = round(max(a) + (1-sum(a)), 100)
            a = np.around(a, 2)
            matrix[i] = a
        if any(np.around(sum(row)) != 1 for row in matrix):
            raise ValueError("failed to normalize matrix after filtering (row does not amount unity).")
        return matrix

    @staticmethod
    def semantic_similarity_number(o1, o2):
        """
        Returns semantic similarity distance value.
        """
        total = 0
        keys  = set(list(o1["word_dict"].keys())+list(o2["word_dict"].keys()))

        for key in keys:

            try: m1 = o1["word_dict"][key]
            except: m1 = 0

            try: m2 = o2["word_dict"][key]
            except: m2 = 0

            total = total + min(m1, m2)

        div = max(o1["total"], o2["total"])

        return (total/div) if div>0 else 0

In [ ]:
pphi = PyPhi()

## Prepare data

#### Input matrix

Load **attention / presence matrix** (`P`) containing a list of states over time, each state being a binary element (`0` for not active; `1` for active).

In [ ]:
P = []

##### Alternative: load matrix from JSON file

Load presence matrix `P` from a previously saved JSON file.

In [ ]:
filename = "matrix.json"

if os.path.isfile(filename):
    P = json.load(open("matrix.json"))

#### Filter connected states (optional)

Optionally get a subset of connected states in a matrix, from a list optionally set by the user (comma separated).

In [ ]:
states_to_remove = "" # <-- comma separated

P, A = pphi.filter_connected_states(P, A=[], states_to_remove=states_to_remove)

#### Check connected states (optional)

Returns a list of connected states based on presence (`P`) and adjacency (`A`) matrices.

In [ ]:
pphi.get_connected_states(A, states=list(P.keys()))

## Render transition diagram

Render a transition diagram from a Markovian chain. Available renderers: `graphviz`, `pygraphviz` and `pydot`.

In [ ]:
states_labels = "" # <-- Comma separated

pphi.render(
    pphi.generate_influence_matrix(list(P.values()), A),
    states=states_labels.split(",") if states_labels else P.keys(),
    output_folder="artifacts",
    renderer="graphviz"
)

## Cause-effect structure

Integrated information theory provides a mathematical framework to fully characterize the cause-effect structure of a physical system.

[PyPhi](http://integratedinformationtheory.org) implements a framework for causal analysis and unfolds the full cause-effect structure of discrete dynamical systems of binary elements.

### Compute network

Returns all complexes in the network context of all φ and Φ computation. Here we’ll use the 2-dimensional state-by-node form for the TPM.

In [ ]:
bc, ac = pphi(P, A)

### Mechanism details

See details (cause and effect) of one selected mechanism, based on the list above.

In [ ]:
m = None # <-- Mechanism number from above (optional)

pphi.complex_mechanism(bc, m)

### Best complex data

Display best complex with the highest phi value after system irreducibility analysis and its cause-effect structure.

In [ ]:
bc

### ALL complexes data (!)

Display all complexes identified by PyPhi. **Warning:** long text buffer.

In [ ]:
ac

## Compress output →  `output.zip`

In [ ]:
!zip -9r output.zip artifacts

_____

## References

* PyPhi: [Website](https://pypi.org/project/pyphi/) | [Paper](https://arxiv.org/abs/1712.09644) | [Documentation](https://pyphi.readthedocs.io/en/latest/) | [Repository](https://github.com/wmayner/pyphi)